<a href="https://colab.research.google.com/github/Natasha-tanzila/Natasha-tanzila/blob/main/00_ingest_raw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a raw file of tables After running the code output file will be cleaned tables ▶

> Reading tables,
> standardized keys,
> standardized time columns,
> basic hygiene for impossible values.





**1.   Reading tables**



* admissions.csv - Core

* edstays.csv - ED

* medrecon.csv - ED  

* patients.csv - Core

* procedures_icd.csv - Core

* pyxis.csv - ED

* triage.csv - ED

* vitalsign.csv - ED







In [ ]:
import pandas as pd

admissions = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/admissions.csv",
    low_memory=False
)

edstays = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/edstays.csv",
    low_memory=False
)

medrecon = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/medrecon.csv",
    low_memory=False
)

patients = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/patients.csv",
    low_memory=False
)

procedures_icd = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/procedures_icd.csv",
    low_memory=False
)

pyxis = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/pyxis.csv",
    low_memory=False
)

triage = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/triage.csv",
    low_memory=False
)

vitalsign = pd.read_csv(
    "/content/drive/MyDrive/D-SATS/Dataset/vitalsign.csv",
    low_memory=False
)

print("✅ All files loaded successfully (absolute paths)")


✅ All files loaded successfully (absolute paths)


In [ ]:
# reading a file

triage.head()

,subject_id,stay_id,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint
0,10000032,32952584,97.8,87.0,14.0,97.0,71.0,43.0,7,2.0,Hypotension
1,10000032,33258284,98.4,70.0,16.0,97.0,106.0,63.0,0,3.0,"Abd pain, Abdominal distention"
2,10000032,35968195,99.4,105.0,18.0,96.0,106.0,57.0,10,3.0,"n/v/d, Abd pain"
3,10000032,38112554,98.9,88.0,18.0,97.0,116.0,88.0,10,3.0,Abdominal distention
4,10000032,39399961,98.7,77.0,16.0,98.0,96.0,50.0,13,2.0,"Abdominal distention, Abd pain, LETHAGIC"


In [ ]:
#inspect current column names

for name, df in {
    "admissions": admissions,
    "edstays": edstays,
    "medrecon": medrecon,
    "patients": patients,
    "procedures_icd": procedures_icd,
    "pyxis": pyxis,
    "triage": triage,
    "vitalsign": vitalsign
}.items():
    print(f"\n{name}")
    print(df.columns.tolist())




admissions
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']

edstays
['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition']

medrecon
['subject_id', 'stay_id', 'charttime', 'name', 'gsn', 'ndc', 'etc_rn', 'etccode', 'etcdescription']

patients
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']

procedures_icd
['subject_id', 'hadm_id', 'seq_num', 'chartdate', 'icd_code', 'icd_version']

pyxis
['subject_id', 'stay_id', 'charttime', 'med_rn', 'name', 'gsn_rn', 'gsn']

triage
['subject_id', 'stay_id', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint']

vitalsign
['subject_id', 'stay_id', 'charttime', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'db

In [ ]:
#inspect current datatypes

tables = {
    "admissions": admissions,
    "edstays": edstays,
    "medrecon": medrecon,
    "patients": patients,
    "procedures_icd": procedures_icd,
    "pyxis": pyxis,
    "triage": triage,
    "vitalsign": vitalsign
}

KEYS = ["subject_id", "hadm_id", "stay_id"]

for name, df in tables.items():
    print(f"\n{name}")
    for k in KEYS:
        if k in df.columns:
            print(f"  {k}: {df[k].dtype}")

#as sequence number is only available in procedures_icd
print("For seq_num's datatype from procedures_icd:")
print(procedures_icd["seq_num"].dtype)




admissions
  subject_id: int64
  hadm_id: int64

edstays
  subject_id: int64
  hadm_id: float64
  stay_id: int64

medrecon
  subject_id: int64
  stay_id: int64

patients
  subject_id: int64

procedures_icd
  subject_id: int64
  hadm_id: int64

pyxis
  subject_id: int64
  stay_id: int64

triage
  subject_id: int64
  stay_id: int64

vitalsign
  subject_id: int64
  stay_id: int64
For seq_num's datatype from procedures_icd:
int64


In [ ]:
# converting all datatypes to Int64 as int64 is not capable of handling missing
# values (like NaN).

import pandas as pd

def enforce_id_dtypes(df):
    """
    Safely enforce nullable integer dtype (Int64)
    for all ID / key-like columns without affecting other columns.
    """
    ID_COLS = ["subject_id", "hadm_id", "stay_id", "seq_num"]

    for col in ID_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    return df


# Apply to all tables
admissions      = enforce_id_dtypes(admissions)
edstays         = enforce_id_dtypes(edstays)
medrecon        = enforce_id_dtypes(medrecon)
patients        = enforce_id_dtypes(patients)
procedures_icd  = enforce_id_dtypes(procedures_icd)
pyxis           = enforce_id_dtypes(pyxis)
triage          = enforce_id_dtypes(triage)
vitalsign       = enforce_id_dtypes(vitalsign)

print("✅ All ID columns safely converted to Int64")


✅ All ID columns safely converted to Int64


In [ ]:
#Sanity check if datatypes are changed
tables = {
    "admissions": admissions,
    "edstays": edstays,
    "medrecon": medrecon,
    "patients": patients,
    "procedures_icd": procedures_icd,
    "pyxis": pyxis,
    "triage": triage,
    "vitalsign": vitalsign
}

KEYS = ["subject_id", "hadm_id", "stay_id"]

for name, df in tables.items():
    print(f"\n{name}")
    for k in KEYS:
        if k in df.columns:
            print(f"  {k}: {df[k].dtype}")

#as sequence number is only available in procedures_icd
print("For seq_num's datatype from procedures_icd:")
print(procedures_icd["seq_num"].dtype)




admissions
  subject_id: Int64
  hadm_id: Int64

edstays
  subject_id: Int64
  hadm_id: Int64
  stay_id: Int64

medrecon
  subject_id: Int64
  stay_id: Int64

patients
  subject_id: Int64

procedures_icd
  subject_id: Int64
  hadm_id: Int64

pyxis
  subject_id: Int64
  stay_id: Int64

triage
  subject_id: Int64
  stay_id: Int64

vitalsign
  subject_id: Int64
  stay_id: Int64
For seq_num's datatype from procedures_icd:
Int64


In [ ]:
#enforcing all timecolumns to be the same as datetime64

import pandas as pd

def enforce_datetime(df):
    """
    Convert all timestamp/date columns to pandas datetime64[ns]
    without affecting non-time columns.
    """
    TIME_COLS = [
        "admittime", "dischtime", "deathtime",
        "edregtime", "edouttime",
        "intime", "outtime",
        "charttime", "chartdate",
        "dod"
    ]

    for col in TIME_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


# Applying to all tables
admissions      = enforce_datetime(admissions)
edstays         = enforce_datetime(edstays)
medrecon        = enforce_datetime(medrecon)
patients        = enforce_datetime(patients)
procedures_icd  = enforce_datetime(procedures_icd)
pyxis           = enforce_datetime(pyxis)
triage          = enforce_datetime(triage)
vitalsign       = enforce_datetime(vitalsign)

print("✅ All time columns converted to datetime64[ns]")



✅ All time columns converted to datetime64[ns]


In [ ]:
#sanity check for all columns datatypes

for name, df in tables.items():
    print(f"\n{name}\n{df.dtypes}")



admissions
subject_id                       Int64
hadm_id                          Int64
admittime               datetime64[ns]
dischtime               datetime64[ns]
deathtime               datetime64[ns]
admission_type                  object
admit_provider_id               object
admission_location              object
discharge_location              object
insurance                       object
language                        object
marital_status                  object
race                            object
edregtime               datetime64[ns]
edouttime               datetime64[ns]
hospital_expire_flag             int64
dtype: object

edstays
subject_id                    Int64
hadm_id                       Int64
stay_id                       Int64
intime               datetime64[ns]
outtime              datetime64[ns]
gender                       object
race                         object
arrival_transport            object
disposition                  object
dtype: object

me

#Section: Handling missing values ⚓



*   I will not do global fill or drop missing values
    

1.   because some missingness id clinically meaningful
2.   Some columns are never to be imputed
3.   Some should be handled later means it is task-specific

So I will handle missing values in 3 tiers

Tier 1:

> I will not touch IDs or primary keys (subject_id, hadm_id, stay_id), outcome/future info (hospital_expire_flag, dod, disposition) and text (chiefcomplaint, medication-name) columns, because if those are missing this is real-world absence not an error.

Tier 2:

> Time missingness means either the event didn't happen or wasn't recorded. So I will never fill timestamps like charttime, admittime, intime, outtime or chartdate. I will keep them as NaT (Not-a-Time) - NaT. when I changed datetime into df[col] = pd.to_datetime(df[col], errors="coerce"), it automatically puts missing timestamps into NaT.


Tier 3:

> Vitals & measurements is where controlled handling makes sense. I will not impute values yet instead flag missingness & optionally create indicators. This preserves information and avoids leakage.



# TIER 3: HANDLE MISSING VALUES


GOAL:
- Inspect missingness
- Preserve clinical reality
- Create missingness indicators (signals)
- NOT imputing or dropping values

WHY:
- Missing vitals/time often mean "not measured"
- This absence itself is informative in ED settings
- Early imputation can introduce bias and leakage

I am not
- ❌ fillna (mean / 0 / forward fill)
- ❌ drop rows
- ❌ so no outcome leakage
- ❌ No timeline interpolation

WHAT WILL BE DONE LATER (Not here):
- ✔ Window-based aggregation (e.g., first 30–60 min)
- ✔ Forward-fill within stay (time-aware)
- ✔ Model-specific imputation (after feature construction)

Window-based aggregation summarises patient condition over an early ED time window (e.g., average/max vitals in the first 30–60 minutes), reducing noise from sparse measurements.

Time-aware forward-fill carries the last observed value forward within the same ED stay, reflecting clinical practice where vitals remain valid until re-measured.

Model-specific imputation is applied only after features are built, allowing different models
to handle remaining missing values appropriately without introducing bias or leakage.



In [ ]:

import pandas as pd

# ---------------------------------------------------------
# STEP 1: INSPECT MISSINGNESS (NO DATA MODIFICATION)
# ---------------------------------------------------------

def missing_summary(df, top_n=10):
    """
    Returns missing ratio per column (descending).
    Purely diagnostic — does NOT modify data.
    """
    return (
        df.isna()
          .mean()
          .sort_values(ascending=False)
          .head(top_n)
          .to_frame("missing_ratio")
    )

print("\n--- Missingness Summary: vitalsign ---")
print(missing_summary(vitalsign))

print("\n--- Missingness Summary: triage ---")
print(missing_summary(triage))

print("\n--- Missingness Summary: medrecon ---")
print(missing_summary(medrecon))

print("\n--- Missingness Summary: pyxis ---")
print(missing_summary(pyxis))


# ---------------------------------------------------------
# STEP 2: CREATE MISSINGNESS INDICATORS FOR VITALS
# ---------------------------------------------------------
# WHY:
# - Missing vitals can indicate severity, workflow delays, or instability
# - Models benefit from knowing that a value was NOT observed
# - This avoids fabricating clinical measurements

VITAL_COLS = [
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp"
]

for col in VITAL_COLS:
    if col in vitalsign.columns:
        vitalsign[f"{col}_missing"] = vitalsign[col].isna().astype("Int64")

print("\n✅ Missingness indicator columns added to vitalsign")
#so missing values will look like: temperature_missing, heartrate_missing,
#resprate_missing etc.




# ---------------------------------------------------------
# STEP 3: TRIAGE — SPECIAL CASE (DO NOTHING)
# ---------------------------------------------------------
# WHY:
# - Triage is a single snapshot at ED entry
# - Missing vitals here mean "not measured at triage"
# - imputing from later vitals (that causes leakage)

# ❌ No action required for triage missingness
print("\nℹ️ Triage missing values intentionally left unchanged")


# ---------------------------------------------------------
# STEP 4: MEDICATION TABLES — DO NOTHING
# ---------------------------------------------------------
# medrecon / pyxis:
# - Missing medication fields mean no medication/event
# - This is clinically meaningful
# - I will not fill or drop

print("\nℹ️ medrecon and pyxis missing values intentionally preserved")


# ---------------------------------------------------------
# FINAL STATE AFTER TIER 3
# ---------------------------------------------------------
# ✔ All missingness preserved
# ✔ Missingness signals added (vitals only)
# ✔ No artificial values introduced
# ✔ No rows removed
# ✔ Safe for downstream temporal modeling

print("\n✅ TIER 3 COMPLETE: Missing values handled safely (no imputation)")



--- Missingness Summary: vitalsign ---
             missing_ratio
rhythm            0.961875
temperature       0.361092
pain              0.283308
o2sat             0.086818
resprate          0.057134
sbp               0.051934
dbp               0.051934
heartrate         0.044554
charttime         0.000000
subject_id        0.000000

--- Missingness Summary: triage ---
                missing_ratio
temperature          0.055083
o2sat                0.048451
resprate             0.047880
dbp                  0.044911
sbp                  0.043029
heartrate            0.040204
pain                 0.030424
acuity               0.016437
chiefcomplaint       0.000054
stay_id              0.000000

--- Missingness Summary: medrecon ---
                missing_ratio
etccode              0.003926
etcdescription       0.003926
subject_id           0.000000
stay_id              0.000000
charttime            0.000000
gsn                  0.000000
name                 0.000000
etc_rn           

In [ ]:
#saving cleaned files to avoid cleaning each time and fasten reloading

from pathlib import Path

# Output directory
CLEANED_DIR = Path("/content/drive/MyDrive/D-SATS/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned tables as CSV ONLY
admissions.to_csv(CLEANED_DIR / "admissions.csv", index=False)
edstays.to_csv(CLEANED_DIR / "edstays.csv", index=False)
medrecon.to_csv(CLEANED_DIR / "medrecon.csv", index=False)
patients.to_csv(CLEANED_DIR / "patients.csv", index=False)
procedures_icd.to_csv(CLEANED_DIR / "procedures_icd.csv", index=False)
pyxis.to_csv(CLEANED_DIR / "pyxis.csv", index=False)
triage.to_csv(CLEANED_DIR / "triage.csv", index=False)
vitalsign.to_csv(CLEANED_DIR / "vitalsign.csv", index=False)

print("✅ Canonical cleaned tables saved as CSV")


✅ Canonical cleaned tables saved as CSV


,temperature_missing,heartrate_missing,resprate_missing,o2sat_missing,sbp_missing,dbp_missing
0,1,0,0,0,0,0
1,1,0,0,0,0,0
2,1,0,0,0,0,0
3,1,0,0,0,0,0
4,0,0,0,0,0,0
